In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

## SIRS Model simplified
The model below is a simplified version of the SIRS model, useful to explain the beginning of the disease's spread - i.e. for small $t$.
The resulting equations are the following:
$$
\begin{cases}
\dot{S} = -\beta SI + \theta R \\
\dot{I} = \beta SI - \gamma I \\
\dot{R} = \gamma I - \theta R
\end{cases}
$$

We focused on the last two equations, so that the resulting model can be written as:
$$
\begin{cases}
\dot{I} = I (\beta - \gamma) -\beta RI - \beta I^{2} \\
\dot{R} = \gamma I - \theta R
\end{cases}
$$

In [ ]:
# TODO (SIRS simplified version)

## SIRS Model

The model studied below is the following:
$$
\begin{cases}
\dot{S} = (\mu + \theta) - (\mu + \theta)S - \beta SI - \theta I \\
\dot{I} = I(\beta S - (\mu + \gamma)) \\
\dot{R} = \gamma I - (\mu + \theta)R
\end{cases}
$$

In particular we focused on the last two equations, so that the resulting model can be written as:
$$
\begin{cases}
\dot{I} = I(\beta - (\mu + \gamma)) - \beta RI - \beta I^{2} \\
\dot{R} = \gamma I - (\mu + \theta)R
\end{cases}
$$

In [ ]:
def SIRS_IR (t, X, beta, gamma, mu, theta):
    I, R = X

    dI = I*(beta - (mu + gamma)) - beta * R * I - beta * I**2
    dR = gamma * I - (mu + theta) * R

    return [dI, dR]

In [ ]:
# Parameters
# These parameters are taken from Grassly 2005 and are related to syphilis cases in the USA

# t = 1 day
# gamma = 0.0167                # Recovery rate
# mu = 0.000083                 # Birth/death rate
# theta = 0.00027               # Loss of immunity rate
# beta = 1.5*(mu+gamma)         # Infection rate

# t = 1 month
gamma = 0.0167*30               # Recovery rate
mu = 0.000083*30                # Birth/death rate
theta = 0.00027*30              # Loss of immunity rate
beta = 1.5*(mu+gamma)           # Infection rate


# Initial conditions
S0 = 0.999                      # Initial suspicious population
I0 = 0.001                      # Initial infected population
R0 = 0                          # Initial recovered population
X0 = [I0, R0]

In [ ]:
# Some specifics of the model
R_0 = round(beta / (mu+gamma), 8)

damping = -(theta + mu)*(theta + beta)/2/(mu + gamma + theta)

period_body = (mu + theta)*(mu + gamma)*(R_0 - 1) - ( (mu + theta)*(theta + beta) / (2*(mu + theta + gamma)) )**2
period = 2*np.pi / np.sqrt(period_body) if period_body > 0 else 0

# this period function is from Keeling, Rohani 20080
# period_si = 4*np.pi / np.sqrt( 4 * (R_0-1) * (mu + gamma) * (mu + theta) - (mu + theta + (mu + theta)*(beta - mu - gamma)/(mu + gamma + theta))**2 )

# Endemic equilibrium
S_e = (mu + gamma) / beta
I_e = (1 - 1/R_0) * (mu + theta) / (mu + gamma + theta)
R_e = (1 - 1/R_0) * gamma / (mu + gamma + theta)

In [ ]:
# print parameters
print(f"Parameters: \nBeta: {beta}\t\t Gamma: {gamma}\t\t Mu: {mu}\nTheta: {theta}\t\tR0: {round(R_0, 3)}\n")
print(f"Specifics: \nDamping: {round(damping, 5)}\t Oscillation period: {round(period, 5)}\nEndemic Equilibrium: {(round(S_e, 3), round(I_e, 3), round(R_e, 3))}")
# print(f"Oscillation SI period: {period_si}")

In [ ]:
# Time span
# t = 1 day
# t_span = (0, 50000)

# t = 1 month
t_span = (0, 1000)

# Solve the system of ODEs
solution = solve_ivp(SIRS_IR, t_span, X0, args=(beta, gamma, mu, theta), dense_output=True)

# Time points for which to get the solution
t = np.linspace(t_span[0], t_span[1], 100000)
sol = solution.sol(t)

In [ ]:
# Impostazione della figura
plt.figure(figsize=(10, 6))

# Tracciamento delle curve
plt.plot(t, 1 - sol[1] - sol[0], label='Suspicious', color='blue')
plt.plot(t, sol[0], label='Infected', color='red')
plt.plot(t, sol[1], label='Recovered', color='green')

# Etichette e titolo
plt.xlabel('Time')
plt.ylabel('Population')
plt.title('SIRS - IR Model')
plt.legend()

# Opzionale: griglia
plt.grid(True)

# Visualizzazione del grafico
plt.show()

In [ ]:
sc = plt.scatter(sol[0], sol[1], c=t, cmap='inferno')  # 'viridis' è un colormap di default
plt.title('Phase plot')
plt.xlabel('Infected')
plt.ylabel('Recovered')
plt.colorbar(sc)  # visualizza la scala dei colori in funzione di t
plt.show()


In [ ]:
sc = plt.scatter(1 - sol[0] - sol[1], sol[0], c=t, cmap='inferno')
plt.title('Phase plot')
plt.xlabel('Suspicious')
plt.ylabel('Infected')
plt.colorbar(sc, label='Time')
plt.show()

## SIRS Model with Human Behaviour
The model studied below is the following:
$$
\begin{cases}
\dot{S} = (\mu + \theta) - (\mu + \theta)S - \beta(M)SI - \theta I \\
\dot{I} = I ( \beta(M)S - (\mu + \gamma) ) \\
\dot{M} = a(I - M)
\end{cases}
$$
, where we arbitrary chose:
$$
\beta(M) = \frac{\beta_{0}}{1 + c M}, \quad
M(t) = \int_{0}^{t} W(\tau)I(t-\tau)d\tau, \quad
W(\tau) = a e^{-a \tau}
$$

In [ ]:
def beta_func(beta_0, c):
    def beta_function(x):
        return beta_0 / (1 + c*x)
    return beta_function

In [ ]:
def SIRS_hb (t, X, beta, gamma, mu, theta, a):
    S, I, M = X

    dS = (mu + theta) - (mu + theta)*S - beta(M)*I*S - theta*I
    dI = I * (beta(M)*S - (mu + gamma) )
    dM = a*(I - M)

    return [dS, dI, dM]

In [ ]:
# parameters

# for now, same as above
# t = 1 month
gamma = 0.0167*30               # Recovery rate
mu = 0.000083*30                # Birth/death rate
theta = 0.00027*30              # Loss of immunity rate
a = 0.0416                      # Memory rate
beta_0 = 2
c = 1
beta = beta_func(beta_0, c)


# Initial conditions
S0 = 0.999                      # Initial suspicious population
I0 = 0.001                      # Initial infected population
M0 = 0                          # Initial recovered population
X0 = [S0, I0, M0]

In [ ]:
1 / 12 / 2

In [ ]:
# t = 1 month
t_span = (0, 1000)

# Solve the system of ODEs
solution = solve_ivp(SIRS_hb, t_span, X0, args=(beta, gamma, mu, theta, a), dense_output=True)

# Time points for which to get the solution
t = np.linspace(t_span[0], t_span[1], 100000)
sol = solution.sol(t)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(t, sol[0], label='Suspicious', color='blue')
plt.plot(t, sol[1], label='Infected', color='red')
plt.plot(t, 1 - sol[0] - sol[1], label='Recovered', color='green')
plt.title('SIRS - Human Behaviour')
plt.xlabel('Time')
plt.ylabel('Population')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(t[0:10000], sol[1][:10000], label='I(t)', color='red')
plt.plot(t[0:10000], sol[2][:10000], label='M(t)', color='magenta')
plt.title('Human Behaviour and Infectious - I(t) and M(t)')
plt.xlabel('Time')
plt.ylabel('Population')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(5, 5))
plt.plot(sol[1], sol[2], label='M(I)', color='blue')
plt.plot(sol[1], sol[1], label='Bisector', color='red')
plt.title('Human Behaviour and Infectious - M(I)')
plt.xlabel('Infectious')
plt.ylabel('M')
plt.legend()
plt.grid(True)
plt.show()